# VLLM AI Juror & Consensus Evaluation Test (`Qwen/Qwen2.5-7B-Instruct`)

This notebook tests the AI Jurors (Compliance, Evidence, HSE) and Consensus Agent using the VLLM REST endpoint (`http://localhost:8000/v1/chat/completions`).

## 1. Setup Environment & Instantiate Jurors

In [ ]:
import os
import sys
import time
import json

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from engine.jurors.compliance_juror import ComplianceJuror
from engine.jurors.evidence_juror import EvidenceJuror
from engine.jurors.hse_juror import HSEJuror
from engine.consensus.consensus_agent import ConsensusAgent

t0 = time.time()
comp_juror = ComplianceJuror(role="reasoning_model")
ev_juror = EvidenceJuror(role="reasoning_model")
hse_juror = HSEJuror(role="reasoning_model")
consensus = ConsensusAgent(role="consensus_model")

print(f"Instantiated Jurors & Consensus in {time.time() - t0:.3f}s")
print(f"Model: {comp_juror.provider.model} | Endpoint: {comp_juror.provider.chat_url}")

## 2. Prepare Sample Evidence & Knowledge Base Context

In [ ]:
sample_category = {
    "category_id": "3.2",
    "title": "Marine Emergency Response & HSE Safety Operations",
    "max_score": 10,
    "required_evidence": ["Emergency Drill Records", "Marine HSE Manual", "Audit Inspection Report"]
}

sample_evidence = {
    "documents": ["Marine_HSE_Manual_v2.pdf", "Drill_Log_2025.pdf"],
    "evidence_snippets": [
        "Emergency response drill executed on 14 Jan 2025 with zero non-conformances.",
        "HSE Marine manual section 4 documents safety management procedures."
    ]
}

sample_validations = {
    "passed": 3,
    "failed": 0,
    "coverage_pct": 100.0
}

## 3. Test Compliance Juror

In [ ]:
t1 = time.time()
comp_res = comp_juror.evaluate(sample_category, sample_evidence, sample_validations)
print(f"[Compliance Juror Evaluated in {time.time() - t1:.3f}s]")
print(json.dumps(comp_res, indent=2))

## 4. Test Evidence Juror

In [ ]:
t2 = time.time()
ev_res = ev_juror.evaluate(sample_category, sample_evidence, sample_validations)
print(f"[Evidence Juror Evaluated in {time.time() - t2:.3f}s]")
print(json.dumps(ev_res, indent=2))

## 5. Test HSE Juror

In [ ]:
t3 = time.time()
hse_res = hse_juror.evaluate(sample_category, sample_evidence, sample_validations)
print(f"[HSE Juror Evaluated in {time.time() - t3:.3f}s]")
print(json.dumps(hse_res, indent=2))

## 6. Test Consensus Agent Resolution

In [ ]:
t4 = time.time()
consensus_res = consensus.resolve(comp_res, ev_res, hse_res, sample_category)
print(f"[Consensus Resolved in {time.time() - t4:.3f}s]")
print(json.dumps(consensus_res, indent=2))